# 판단 모델 judge 파일럿

판단 모델(Decisions API) 세 개가 지금 judge(j_v8, DeepSeek V4.1 Flash)의 판정 단계를 대신할 수 있는지, 이미 있는 사람 라벨과의 일치로 본다. 대상 단계는 `item_handling`, `relevance`, `verdicts`다. 모델은 `typesafe/jev-1.14`(기준), `microsoft/microsoft-decision-1`, `openai/gpt-6-luna-decisions`다(사용자 지시 2026-10-26).

- 사람 라벨은 judge 지시를 만들 때 쓴 개발 데이터(S14 표본, S14c 확인 사례)다. 여기서 나온 일치율은 모델과 단계를 고르는 근거로만 쓰고 검증 근거로 쓰지 않는다(기준 문서 P3)
- 판단 모델에 넣는 지시는 j_v8의 규칙을 질문·선택지 설명으로 옮긴 것이다. 예시는 넣지 않았다
- 질의 원문은 출력에 남기지 않고 qid·사례 번호와 집계만 남긴다(D-31)

In [1]:
import hashlib
import json
import os
import time
import urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langsmith import traceable
from langsmith.run_trees import get_cached_client

from rag.gen_eval.checks import item_outcome
from rag.gen_eval.judge import Judge, faithfulness, format_contexts, relevance
from rag.index import load_cfg

In [2]:
ROOT = Path.cwd().parent
load_dotenv(ROOT / ".env")
CASES = ROOT / "data/gen_eval/s14c_cases.jsonl"
S14_RUN = ROOT / "data/gen_eval/runs/hybrid-cond-filter-law-ms256-k20-ce-v1-dev-small.jsonl"
S14_REVIEW = ROOT / "data/review/s14_judge_think.csv"
CACHE = ROOT / "data/gen_eval/decision_cache"  # 요청 본문 해시로 판단 모델 응답을 저장한다
URL = "https://openrouter.ai/api/alpha/decisions"
MODELS = ["typesafe/jev-1.14", "microsoft/microsoft-decision-1", "openai/gpt-6-luna-decisions"]
WORKERS = 8
NOUL_MIN = 0.5  # 뒷받침 확률이 이 값 이상이면 1로 본다

# j_v8 item_handling의 처리 정의를 선택지 설명으로 옮겼다
HANDLING_CRITERIA = {
    "답함": "이 항목에 답했다. 검토·예정을 확정된 계획이나 사실처럼 단정해 쓴 것도 여기에 든다.",
    "부분임을 밝히고 답함": "이 항목에 답하면서, 회의록에 일부만 있다거나 나머지는 찾지 못했다고 밝혔다.",
    "상황을 밝히고 답함": "확정된 답 대신, 회의에서 검토하겠다·예정이다·정해지지 않았다·협의하겠다고만 했다거나 "
                     "답변자가 답을 피했다는 상황을 전했다. 일부만 있다는 안내와 함께 이런 상황을 전한 것도 여기에 든다.",
    "거부": "이 항목은 회의록에서 답을 찾지 못했다는 취지로 밝히고 답하지 않았다.",
    "다루지 않음": "이 항목에 답하지도, 찾지 못했다고 밝히지도 않았다.",
}
# j_v8 relevance의 판정 정의. 순서 척도라 낮은 단계부터 둔다
RELEVANCE_LEVELS = [
    "답하지 않음: 질문과 다른 쟁점을 말하거나 묻는 것에 답이 없다.",
    "일부: 질문이 묻는 것 중 일부만 답하거나, 관련된 이야기를 하지만 묻는 것에 정면으로 답하지 않는다.",
    "답함: 질문의 요지에 실질적으로 답한다. 여러 가지를 물으면 모두 답한다. 근거의 관련 입장을 정리해 요지에 답하거나, "
    "불가능하다는 것을 밝히거나, 확답이 없었다는 상황(검토하겠다, 답을 피했다)을 전한 것도 답함이다. "
    "찾지 못했다고 밝힌 부분은 감점하지 않는다.",
]
RELEVANCE_INSTRUCTIONS = "답변이 질문에 답하는가? 답변 내용이 사실인지는 보지 않는다."
# j_v8 verdicts 규칙 1~5를 예·아니오 설명으로 옮겼다
VERDICT_CRITERIA = {
    "true": "진술이 근거에 쓰여 있거나 근거에서 바로 따라 나온다. 숫자·금액·날짜는 표기만 다르면 같은 내용이다. "
            "'검토하겠습니다'처럼 답을 피한 발언도 근거에 있으면 뒷받침된다. 질의자의 주장을 '○○ 위원은 ~라고 "
            "주장했다'처럼 그 주장으로 쓴 것도 뒷받침된다. 발언자 표기가 달라도 내용이 근거에 있으면 뒷받침된다.",
    "false": "진술이 근거에 없거나 상식·추측으로 채운 내용이다. 질의자(위원)가 질문하거나 주장했을 뿐 답변자가 "
             "확인하지 않은 내용을 사실처럼 쓴 것, 질의자 주장과 답변자의 '검토하겠다'를 묶어 답변자가 계획을 "
             "분명히 밝힌 것처럼 쓴 것도 뒷받침되지 않는다.",
}

In [3]:
def sha(obj) -> str:
    return hashlib.sha256(json.dumps(obj, ensure_ascii=False, sort_keys=True).encode()).hexdigest()


@traceable(run_type="llm", name="decision_judge", process_inputs=lambda i: {"model": i["body"]["model"]})
def post(body: dict) -> dict:
    req = urllib.request.Request(URL, data=json.dumps(body).encode(), headers={
        "Authorization": f"Bearer {os.environ['OPENROUTER_API']}", "Content-Type": "application/json"})
    t = time.time()
    with urllib.request.urlopen(req, timeout=120) as r:
        resp = json.loads(r.read().decode())
    return {**resp, "latency_s": time.time() - t}


def decide(body: dict) -> dict:
    """같은 요청 본문은 저장된 응답을 쓴다. 오류는 저장하지 않고 {'error': ...}로 돌려준다."""
    path = CACHE / f"{sha(body)}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    try:
        resp = post(body)
    except Exception as e:  # 모델 하나의 실패는 그 모델의 결과로 기록한다
        return {"error": f"{type(e).__name__}: {e}"}
    path.write_text(json.dumps(resp, ensure_ascii=False), encoding="utf-8")
    return resp


def run_all(bodies: list[dict]) -> list[dict]:
    with ThreadPoolExecutor(WORKERS) as ex:
        out = list(ex.map(decide, bodies))
    get_cached_client().flush()
    return out

## 데이터 로딩

In [4]:
CACHE.mkdir(parents=True, exist_ok=True)
cases = [json.loads(line) for line in CASES.open(encoding="utf-8")]
items = [{"case": c["case"], "k": k, **e} for c in cases for k, e in enumerate(c["expected"])]
print(f"사례 {len(cases)}개, 정답 항목 {len(items)}개")
print("처리 정답 분포", Counter(i["handling"] for i in items))
print("정보 상태 정답 분포", Counter(i["state"] for i in items))
print("정답 출처", Counter(i["source"] for i in items))

사례 20개, 정답 항목 41개
처리 정답 분포 Counter({'답함': 18, '거부': 11, '다루지 않음': 5, '상황을 밝히고 답함': 5, '부분임을 밝히고 답함': 2})
정보 상태 정답 분포 Counter({'있음': 21, '모호': 11, '없음': 6, '일부': 3})
정답 출처 Counter({'사용자': 26, '메인': 9, '사용자(judge 1회차 뒤 고침)': 6})


In [6]:
gens = {r["qid"]: r for r in map(json.loads, S14_RUN.open(encoding="utf-8"))}
review = pd.read_csv(S14_REVIEW, encoding="utf-8", dtype=str)


def human_value(r) -> str | None:
    """사람 판정 값. 동의·켬·끔은 고른 쪽 judge 값, 둘 다 틀림은 사용자가 쓴 값이다."""
    choice = r["user_choice"]
    if choice in ("동의", "켬"):
        return r["deepseek_go"]
    if choice == "끔":
        return r["deepseek_go_nothink"]
    if choice == "둘 다 틀림":
        return r["user_score"]
    return None


review["human"] = review.apply(human_value, axis=1)
review["by_user"] = review["filled_by"].str.startswith("user")
labels = review[review["criterion"].isin(["충실성", "답변 관련성"])][
    ["qid", "criterion", "human", "by_user"]]
for crit, g in labels.groupby("criterion"):
    print(crit, g["human"].value_counts(dropna=False).to_dict(), "사용자 직접", int(g["by_user"].sum()))
print("생성 기록", len(gens), "거부 문구 답변", sum("찾지 못했습니다" in g["answer"] for g in gens.values()))

답변 관련성 {'1': 17, '0.5': 3} 사용자 직접 5
충실성 {'1': 14, nan: 1, '0.43': 1, '0.83': 1, '0.88': 1, '0': 1, '0.8': 1} 사용자 직접 5
생성 기록 20 거부 문구 답변 0


## 기준선: 지금 judge(j_v8)

v1 생성 기록은 그 뒤 바뀐 프롬프트 파일과 달라 `rag.gen_eval.judge`가 채점 전에 멈춘다(S16 확인). 판정 함수(`faithfulness`, `relevance`)를 직접 불러 j_v8로 채점한다. 진술 추출(`statements`)은 j_v5와 글자까지 같아 저장된 응답을 쓴다. 거부 문구 답변이 없어 거부 문장 처리는 필요 없다.

In [7]:
cfg = load_cfg(["index.scope=full", "+exp=exp017"])
print(cfg.judge.name, cfg.judge.prompt.version)
judge = Judge(cfg, session="meet-rag-decision-pilot")
qids = sorted(gens)


def base_one(q):
    g = gens[q]
    return q, faithfulness(judge, g["query"], g["answer"], g["contexts"]), relevance(judge, g["query"], g["answer"])


with ThreadPoolExecutor(WORKERS) as ex:
    base = {q: (f, r) for q, f, r in ex.map(base_one, qids)}
get_cached_client().flush()
print(f"API 호출 {judge.calls}회, 저장된 응답 {judge.hits}회")
print("형식 오류", sum(1 for f, r in base.values() if f.get("score") is None or r.get("score") is None))

deepseek_go_mixed j_v8
API 호출 38회, 저장된 응답 22회
형식 오류 0


In [8]:
def base_handling(c):
    listed = [{"n": k, "item": e["item"]} for k, e in enumerate(c["expected"], 1)]
    out, _ = judge.ask("item_handling", question=c["query"], answer=c["answer"],
                       items=json.dumps(listed, ensure_ascii=False))
    got = (out or {}).get("handling") or []
    return c["case"], [h.get("handling") for h in got] if len(got) == len(listed) else None


with ThreadPoolExecutor(WORKERS) as ex:
    base_h = dict(ex.map(base_handling, cases))
get_cached_client().flush()
print(f"API 호출 누계 {judge.calls}회, 형식 오류 사례 {sum(v is None for v in base_h.values())}")

API 호출 누계 56회, 형식 오류 사례 0


## 판단 모델 호출

- `item_handling`: 사례마다 한 요청. 상태는 질문·답변이고, 정답 항목마다 처리 선택지 질문 하나를 둔다(근거는 넣지 않는다, j_v8과 같음)
- `relevance`: 질의마다 순서 척도 질문 하나. 상태는 질문·답변
- `verdicts`: 질의마다 한 요청. 상태는 근거 5개, 진술마다 예·아니오 질문 하나. 진술은 j_v8 기준선이 쓴 진술과 같다

In [9]:
def handling_body(m, c):
    qs = {f"i{k}": {"type": "choice", "criteria": HANDLING_CRITERIA,
                    "instructions": f"답변은 질문의 항목 '{e['item']}'을 어떻게 처리했는가? 답변 내용이 사실인지는 보지 않는다."}
          for k, e in enumerate(c["expected"])}
    return {"model": m, "state": {"질문": c["query"], "답변": c["answer"]}, "questions": qs}


def relevance_body(m, q):
    g = gens[q]
    return {"model": m, "state": {"질문": g["query"], "답변": g["answer"]},
            "questions": {"rel": {"type": "score", "instructions": RELEVANCE_INSTRUCTIONS,
                                  "criteria": RELEVANCE_LEVELS}}}


def verdict_body(m, q):
    st = [v["statement"] for v in base[q][0]["verdicts"]]
    qs = {f"s{k}": {"type": "noul", "criteria": VERDICT_CRITERIA,
                    "instructions": f"진술 '{s}'이 회의록 근거로 뒷받침되는가?"} for k, s in enumerate(st)}
    return {"model": m, "state": {"근거": format_contexts(gens[q]["contexts"])}, "questions": qs}


jobs = ([("handling", m, c["case"], handling_body(m, c)) for m in MODELS for c in cases]
        + [("relevance", m, q, relevance_body(m, q)) for m in MODELS for q in qids]
        + [("verdicts", m, q, verdict_body(m, q)) for m in MODELS for q in qids])
resps = run_all([b for *_, b in jobs])
res = {(t, m, key): r for (t, m, key, _), r in zip(jobs, resps, strict=True)}
errors = Counter((t, m) for (t, m, _), r in res.items() if "error" in r)
print(f"요청 {len(jobs)}개, 오류 {dict(errors) or 0}")
for (t, m, k), r in res.items():
    if "error" in r:
        print(t, m, k, r["error"][:200])
        break

요청 180개, 오류 0


## 비교: item_handling (S14c 정답 항목)

맞는 처리 여부는 정답의 정보 상태와 각 모델의 처리로 판정표(I8)를 적용해 정답의 맞는 처리 여부와 비교한다.

In [10]:
def model_handling(m, c):
    r = res["handling", m, c["case"]]
    if "error" in r:
        return None
    return [r["answers"][f"i{k}"]["choice"] for k in range(len(c["expected"]))]


getters = [("j_v8 (DeepSeek)", lambda c: base_h[c["case"]])] + [
    (m, lambda c, m=m: model_handling(m, c)) for m in MODELS]
rows = []
for name, get in getters:
    same_h = same_o = n = miss = 0
    for c in cases:
        got = get(c)
        if got is None:
            miss += len(c["expected"])
            continue
        for e, h in zip(c["expected"], got, strict=True):
            n += 1
            same_h += h == e["handling"]
            same_o += item_outcome(e["state"], h) == item_outcome(e["state"], e["handling"])
    rows.append({"모델": name, "항목": n, "처리 일치": f"{same_h}/{n}",
                 "맞는 처리 여부 일치": f"{same_o}/{n}", "판단 불가 항목": miss})
pd.DataFrame(rows)

,모델,항목,처리 일치,맞는 처리 여부 일치,판단 불가 항목
0,j_v8 (DeepSeek),41,39/41,39/41,0
1,typesafe/jev-1.14,41,34/41,35/41,0
2,microsoft/microsoft-decision-1,41,38/41,38/41,0
3,openai/gpt-6-luna-decisions,41,33/41,33/41,0


In [11]:
# 어느 처리에서 어긋나는지: (정답 처리, 모델 처리) 쌍의 수
for name, get in getters:
    pairs = Counter((e["handling"], h) for c in cases if get(c)
                    for e, h in zip(c["expected"], get(c), strict=True) if e["handling"] != h)
    print(name, dict(pairs))

j_v8 (DeepSeek) {('부분임을 밝히고 답함', '답함'): 2}
typesafe/jev-1.14 {('다루지 않음', '답함'): 3, ('부분임을 밝히고 답함', '상황을 밝히고 답함'): 1, ('답함', '상황을 밝히고 답함'): 1, ('거부', '상황을 밝히고 답함'): 1, ('다루지 않음', '부분임을 밝히고 답함'): 1}
microsoft/microsoft-decision-1 {('다루지 않음', '답함'): 2, ('부분임을 밝히고 답함', '상황을 밝히고 답함'): 1}
openai/gpt-6-luna-decisions {('다루지 않음', '답함'): 2, ('상황을 밝히고 답함', '답함'): 3, ('부분임을 밝히고 답함', '상황을 밝히고 답함'): 1, ('상황을 밝히고 답함', '부분임을 밝히고 답함'): 1, ('다루지 않음', '부분임을 밝히고 답함'): 1}


## 비교: relevance (S14 v1 답변, 사람 라벨)

사람 라벨은 2026-10-20 사용자 기준이다. 그 뒤 D-47(확답 없음을 전한 답변은 답함, 부분 거부의 찾지 못한 부분은 감점 안 함)이 바뀌었으므로 그 경우의 라벨은 지금 규칙과 다를 수 있다.

In [12]:
LEVEL_VALUE = {0: 0.0, 1: 0.5, 2: 1.0}
rel_h = labels[labels["criterion"] == "답변 관련성"].dropna(subset=["human"]).set_index("qid")


def model_relevance(m, q):
    r = res["relevance", m, q]
    if "error" in r:
        return None
    p = r["answers"]["rel"]["probabilities"]
    return LEVEL_VALUE[int(max(p, key=p.get))]


rows = []
for name, get in [("j_v8 (DeepSeek)", lambda q: base[q][1]["score"])] + [
        (m, lambda q, m=m: model_relevance(m, q)) for m in MODELS]:
    vals = {q: get(q) for q in rel_h.index}
    ok = [q for q, v in vals.items() if v is not None]
    same = [vals[q] == float(rel_h.loc[q, "human"]) for q in ok]
    user = [vals[q] == float(rel_h.loc[q, "human"]) for q in ok if rel_h.loc[q, "by_user"]]
    rows.append({"모델": name, "질의": len(ok), "사람과 같음": f"{sum(same)}/{len(same)}",
                 "사용자 직접 판정 행만": f"{sum(user)}/{len(user)}",
                 "분포": dict(Counter(vals[q] for q in ok))})
print("사람 라벨 분포", dict(Counter(rel_h["human"])))
pd.DataFrame(rows)

사람 라벨 분포 {'0.5': 3, '1': 17}


,모델,질의,사람과 같음,사용자 직접 판정 행만,분포
0,j_v8 (DeepSeek),20,17/20,3/5,"{0.5: 2, 1.0: 18}"
1,typesafe/jev-1.14,20,17/20,3/5,{1.0: 20}
2,microsoft/microsoft-decision-1,20,17/20,3/5,{1.0: 20}
3,openai/gpt-6-luna-decisions,20,16/20,3/5,"{0.5: 3, 1.0: 17}"


## 비교: verdicts (S14 v1 답변)

진술 단위 사람 라벨은 없다. 질의 단위 충실성(뒷받침 진술 비율)을 사람 값과 비교하고, 진술 단위로는 j_v8 판정과의 일치만 본다(j_v8은 사람 라벨이 아니다).

In [13]:
faith_h = labels[labels["criterion"] == "충실성"].dropna(subset=["human"]).set_index("qid")


def model_verdicts(m, q):
    r = res["verdicts", m, q]
    if "error" in r:
        return None
    a = r["answers"]
    return [a[f"s{k}"]["noul"] for k in range(len(a))]


rows = []
for name, get in [("j_v8 (DeepSeek)", lambda q: [v["verdict"] for v in base[q][0]["verdicts"]])] + [
        (m, lambda q, m=m: model_verdicts(m, q)) for m in MODELS]:
    err, agree, n, zeros, n_q = [], 0, 0, 0, 0
    for q in qids:
        got = get(q)
        if got is None:
            continue
        n_q += 1
        bits = [int(p >= NOUL_MIN) for p in got]
        ref = [v["verdict"] for v in base[q][0]["verdicts"]]
        agree += sum(b == x for b, x in zip(bits, ref, strict=True))
        n += len(bits)
        zeros += bits.count(0)
        if q in faith_h.index and faith_h.loc[q, "human"] != "-":
            err.append(abs(np.mean(bits) - float(faith_h.loc[q, "human"])))
    rows.append({"모델": name, "질의": n_q, "질의 충실성 |차| 평균(사람)": f"{np.mean(err):.3f} ({len(err)}질의)",
                 "사람과 같은 질의": f"{sum(e < 1e-9 for e in err)}/{len(err)}",
                 "진술 j_v8 일치": f"{agree}/{n}", "0 판정 진술": zeros})
pd.DataFrame(rows)

,모델,질의,질의 충실성 |차| 평균(사람),사람과 같은 질의,진술 j_v8 일치,0 판정 진술
0,j_v8 (DeepSeek),20,0.078 (19질의),13/19,64/64,15
1,typesafe/jev-1.14,20,0.101 (19질의),14/19,49/64,2
2,microsoft/microsoft-decision-1,20,0.101 (19질의),14/19,50/64,1
3,openai/gpt-6-luna-decisions,20,0.108 (19질의),14/19,48/64,1


In [14]:
# 판단 모델의 뒷받침 확률 분포(경계 사례가 얼마나 있는지)
for m in MODELS:
    ps = [p for q in qids if model_verdicts(m, q) for p in model_verdicts(m, q)]
    print(m, "진술", len(ps), "구간 [0,.1,.3,.5,.7,.9,1]",
          np.histogram(ps, bins=[0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0001])[0].tolist())

typesafe/jev-1.14 진술 64 구간 [0,.1,.3,.5,.7,.9,1] [0, 2, 0, 1, 5, 56]
microsoft/microsoft-decision-1 진술 64 구간 [0,.1,.3,.5,.7,.9,1] [1, 0, 0, 0, 3, 60]
openai/gpt-6-luna-decisions 진술 64 구간 [0,.1,.3,.5,.7,.9,1] [0, 0, 1, 0, 1, 62]


In [16]:
# 실패 클래스: 사람이 충실성을 1보다 낮게 준 질의에서 각 모델의 질의 충실성
low = [q for q in faith_h.index if faith_h.loc[q, "human"] not in ("-", "1")]
table = {"사람": {q: float(faith_h.loc[q, "human"]) for q in low},
         "j_v8 (DeepSeek)": {q: base[q][0]["score"] for q in low}}
for m in MODELS:
    table[m] = {q: float(np.mean([int(p >= NOUL_MIN) for p in model_verdicts(m, q)])) for q in low}
pd.DataFrame(table).round(2)

,사람,j_v8 (DeepSeek),typesafe/jev-1.14,microsoft/microsoft-decision-1,openai/gpt-6-luna-decisions
049893-0006,0.43,0.43,1.00,1.00,1.0
052269-0014,0.83,0.17,1.00,1.00,1.0
043451-0001,0.88,0.75,1.00,1.00,1.0
047751-0017,0.00,0.00,1.00,1.00,1.0
051553-0016,0.80,0.75,0.75,0.75,1.0


## 비용·지연

In [15]:
rows = []
for m in MODELS:
    rs = [r for (t, mm, _), r in res.items() if mm == m and "error" not in r]
    lat = [r["latency_s"] for r in rs if "latency_s" in r]
    rows.append({"모델": m, "요청": len(rs), "입력 토큰": sum(r["usage"]["input_tokens"] for r in rs),
                 "비용 $": round(sum(r["usage"].get("cost") or 0 for r in rs), 5),
                 "지연 p50 s": round(float(np.median(lat)), 2) if lat else None,
                 "지연 p95 s": round(float(np.percentile(lat, 95)), 2) if lat else None})
pd.DataFrame(rows)

,모델,요청,입력 토큰,비용 $,지연 p50 s,지연 p95 s
0,typesafe/jev-1.14,60,111494,0.00468,0.42,0.72
1,microsoft/microsoft-decision-1,60,64407,0.00271,0.74,1.63
2,openai/gpt-6-luna-decisions,60,89326,0.00893,0.50,0.77


## 관찰

- `verdicts`: 판단 모델 세 개는 진술 64개 중 1\~2개만 뒷받침 안 됨으로 판정했다(j_v8은 15개). 사람이 충실성을 1보다 낮게 준 5질의 중 판단 모델은 4\~5질의를 1.00으로 냈고, 사람이 0을 준 047751-0017도 세 모델 모두 1.00이었다. 질의 충실성이 사람과 같은 질의 수(14/19)가 j_v8(13/19)보다 많은 것은 사람 라벨 19개 중 14개가 1이라, 모두 1로 내도 맞기 때문이다. 근거 없는 진술을 잡는 실패 클래스에서 판단 모델은 거의 잡지 못했다
- `relevance`: 사람 라벨 20개 중 17개가 답함이라 구분력을 보기 어렵다. Jev 1.14와 Microsoft-Decision-1은 20개 모두 답함으로 냈다
- `item_handling`: 정답 41항목 중 처리 일치는 j_v8 39, Microsoft-Decision-1 38, Jev 1.14 34, GPT-6 Luna Decisions 33이다. j_v8 지시는 이 사례들을 보며 고친 지시라 j_v8 쪽에 유리하다. 판단 모델의 공통 오류는 "다루지 않음"을 "답함"으로 본 것이다
- 비용: 판단 모델 180요청 합계 약 $0.016. 지연 p50은 0.42\~0.74초다

## 2차: solar-decide와 판단 모델용 verdicts 지시

사용자 지시(2026-10-26): 모델을 `upstage/solar-decide`로 바꾸고, `verdicts` 지시를 판단 모델에 맞춰 고쳐 다시 본다.

- solar-decide는 1차와 같은 지시로 세 단계를 모두 본다
- 판단 모델용 `verdicts`(v2)는 OpenRouter Decisions 문서의 한계 목록(`decision-model-limits.md`)을 따랐다. 한 질문에 판단 하나만 두라는 권고에 따라 진술마다 세 질문으로 나눈다. ① 진술 내용이 근거에서 확인된다 ② 진술 내용이 근거에서 위원의 질문·주장으로만 나오고 답변자가 확인하지 않았다 ③ 진술이 그 내용을 위원·단체의 주장으로 소개한다. 결합은 코드에서 한다: ① ≥ 0.5이고 (② < 0.5이거나 ③ ≥ 0.5)이면 뒷받침(1)이다
- v2는 지시 효과와 모델 차이를 가르려고 1차의 세 모델에도 같이 돌린다

In [17]:
SOLAR = "upstage/solar-decide"
ALL_MODELS = MODELS + [SOLAR]
# 판단 모델용 verdicts(v2). 진술마다 세 질문, 결합은 코드에서 한다
V2_QUESTIONS = {
    "in": ("이 진술의 내용은 근거 회의록에서 확인된다.",
           {"true": "근거 회의록에 같은 내용이 쓰여 있거나 근거에서 바로 따라 나온다. 숫자·날짜는 표기만 달라도 같은 내용이다.",
            "false": "근거 회의록에 없는 내용이나 근거를 넘어선 추측이 들어 있다."}),
    "q_only": ("이 진술의 내용은 근거에서 위원(질의자)의 질문이나 주장으로만 나오고, 답변자(장관·기관장 등)는 그 내용을 확인하지 않았다.",
               {"true": "그 내용은 위원이 묻거나 주장한 것뿐이고, 답변자의 발언에서는 확인되지 않는다.",
                "false": "그 내용은 답변자의 발언이나 확정된 사실로 근거에 나온다."}),
    "attr": ("이 진술은 그 내용을 특정 위원이나 단체의 주장·질문으로 소개한다.",
             {"true": "'○○ 위원은 ~라고 주장했다', '○○은 ~라고 물었다'처럼 누구의 주장·질문인지 밝혀 쓴다.",
              "false": "그 내용을 사실이나 답변자의 입장처럼 쓴다."}),
}


def verdict_v2_body(m, q):
    st = [v["statement"] for v in base[q][0]["verdicts"]]
    qs = {f"s{k}_{key}": {"type": "noul", "instructions": f"진술: '{s}'\n{text}", "criteria": crit}
          for k, s in enumerate(st) for key, (text, crit) in V2_QUESTIONS.items()}
    return {"model": m, "state": {"근거": format_contexts(gens[q]["contexts"])}, "questions": qs}


jobs2 = ([("handling", SOLAR, c["case"], handling_body(SOLAR, c)) for c in cases]
         + [("relevance", SOLAR, q, relevance_body(SOLAR, q)) for q in qids]
         + [("verdicts", SOLAR, q, verdict_body(SOLAR, q)) for q in qids]
         + [("verdicts_v2", m, q, verdict_v2_body(m, q)) for m in ALL_MODELS for q in qids])
resps2 = run_all([b for *_, b in jobs2])
res |= {(t, m, key): r for (t, m, key, _), r in zip(jobs2, resps2, strict=True)}
errors2 = Counter((t, m) for (t, m, _, _), r in zip(jobs2, resps2, strict=True) if "error" in r)
print(f"요청 {len(jobs2)}개, 오류 {dict(errors2) or 0}")
for (t, m, k, _), r in zip(jobs2, resps2, strict=True):
    if "error" in r:
        print(t, m, k, r["error"][:300])
        break

요청 140개, 오류 0


In [18]:
# solar-decide, 1차와 같은 지시: item_handling과 relevance
h_same = h_out = n = 0
pairs = Counter()
for c in cases:
    got = model_handling(SOLAR, c)
    for e, h in zip(c["expected"], got, strict=True):
        n += 1
        h_same += h == e["handling"]
        h_out += item_outcome(e["state"], h) == item_outcome(e["state"], e["handling"])
        if h != e["handling"]:
            pairs[e["handling"], h] += 1
print(f"item_handling 처리 일치 {h_same}/{n}, 맞는 처리 여부 일치 {h_out}/{n}, 어긋난 쌍 {dict(pairs)}")
vals = {q: model_relevance(SOLAR, q) for q in rel_h.index}
same = sum(vals[q] == float(rel_h.loc[q, "human"]) for q in rel_h.index)
user = [vals[q] == float(rel_h.loc[q, "human"]) for q in rel_h.index if rel_h.loc[q, "by_user"]]
print(f"relevance 사람과 같음 {same}/{len(rel_h)}, 사용자 직접 {sum(user)}/{len(user)}, 분포 {dict(Counter(vals.values()))}")

item_handling 처리 일치 35/41, 맞는 처리 여부 일치 35/41, 어긋난 쌍 {('다루지 않음', '답함'): 2, ('부분임을 밝히고 답함', '상황을 밝히고 답함'): 1, ('거부', '부분임을 밝히고 답함'): 1, ('상황을 밝히고 답함', '부분임을 밝히고 답함'): 1, ('다루지 않음', '부분임을 밝히고 답함'): 1}
relevance 사람과 같음 15/20, 사용자 직접 3/5, 분포 {1.0: 18, 0.5: 2}


In [19]:
# verdicts: 1차 지시(v1)와 판단 모델용 지시(v2)의 진술 판정
def v2_bits(m, q):
    a = res["verdicts_v2", m, q]["answers"]
    k_n = len(base[q][0]["verdicts"])
    return [int(a[f"s{k}_in"]["noul"] >= NOUL_MIN
                and (a[f"s{k}_q_only"]["noul"] < NOUL_MIN or a[f"s{k}_attr"]["noul"] >= NOUL_MIN))
            for k in range(k_n)]


def v1_bits(m, q):
    return [int(p >= NOUL_MIN) for p in model_verdicts(m, q)]


ref_bits = {q: [v["verdict"] for v in base[q][0]["verdicts"]] for q in qids}
variants = [("j_v8 (DeepSeek)", lambda q: ref_bits[q])] + [
    (f"{m} v1", lambda q, m=m: v1_bits(m, q)) for m in ALL_MODELS] + [
    (f"{m} v2", lambda q, m=m: v2_bits(m, q)) for m in ALL_MODELS]
rows = []
for name, get in variants:
    bits = {q: get(q) for q in qids}
    err = [abs(np.mean(bits[q]) - float(faith_h.loc[q, "human"])) for q in faith_h.index if faith_h.loc[q, "human"] != "-"]
    rows.append({"설정": name, "0 판정 진술": sum(b.count(0) for b in bits.values()),
                 "진술 j_v8 일치": f"{sum(x == y for q in qids for x, y in zip(bits[q], ref_bits[q], strict=True))}/{sum(map(len, ref_bits.values()))}",
                 "질의 충실성 |차| 평균(사람)": round(float(np.mean(err)), 3)})
pd.DataFrame(rows)

,설정,0 판정 진술,진술 j_v8 일치,질의 충실성 |차| 평균(사람)
0,j_v8 (DeepSeek),15,64/64,0.078
1,typesafe/jev-1.14 v1,2,49/64,0.101
2,microsoft/microsoft-decision-1 v1,1,50/64,0.101
3,openai/gpt-6-luna-decisions v1,1,48/64,0.108
4,upstage/solar-decide v1,1,48/64,0.108
5,typesafe/jev-1.14 v2,8,45/64,0.191
6,microsoft/microsoft-decision-1 v2,22,43/64,0.290
7,openai/gpt-6-luna-decisions v2,2,47/64,0.101
8,upstage/solar-decide v2,11,38/64,0.284


In [20]:
# 실패 클래스(사람 충실성 < 1) 질의에서 v2의 질의 충실성과 2차 비용
table2 = {"사람": table["사람"], "j_v8 (DeepSeek)": table["j_v8 (DeepSeek)"]}
table2[f"{SOLAR} v1"] = {q: float(np.mean(v1_bits(SOLAR, q))) for q in low}
for m in ALL_MODELS:
    table2[f"{m} v2"] = {q: float(np.mean(v2_bits(m, q))) for q in low}
display(pd.DataFrame(table2).round(2).T)
cost2 = Counter()
for (t, m, _, _), r in zip(jobs2, resps2, strict=True):
    cost2[m] += r["usage"].get("cost") or 0
print("2차 비용 $", {m: round(v, 5) for m, v in cost2.items()}, "합계", round(sum(cost2.values()), 4))

,049893-0006,052269-0014,043451-0001,047751-0017,051553-0016
사람,0.43,0.83,0.88,0.0,0.80
j_v8 (DeepSeek),0.43,0.17,0.75,0.0,0.75
upstage/solar-decide v1,1.00,1.00,1.00,1.0,1.00
typesafe/jev-1.14 v2,0.86,0.67,0.75,1.0,1.00
microsoft/microsoft-decision-1 v2,0.57,0.67,0.75,0.0,0.00
openai/gpt-6-luna-decisions v2,0.86,1.00,1.00,1.0,1.00
upstage/solar-decide v2,1.00,1.00,0.75,1.0,1.00


2차 비용 $ {'upstage/solar-decide': 0.02483, 'typesafe/jev-1.14': 0.00362, 'microsoft/microsoft-decision-1': 0.00237, 'openai/gpt-6-luna-decisions': 0.00901} 합계 0.0398


## 2차 관찰

- solar-decide(1차 지시): `item_handling` 35/41, `relevance` 15/20으로 1차의 Microsoft-Decision-1(38/41, 17/20)보다 낮다. `verdicts`는 다른 판단 모델과 같이 진술 64개 중 1개만 0으로 판정했다
- 판단 모델용 `verdicts`(v2)는 0 판정 수를 바꿨지만 맞는 곳에서 바꾸지 못했다. Microsoft-Decision-1 v2는 0 판정이 22개로 늘어 사람이 0을 준 047751-0017을 0으로 잡았지만, 사람이 0.80을 준 051553-0016도 0.00으로 냈다. 진술 단위 j_v8 일치는 43/64로 1차(50/64)보다 낮고, 질의 충실성의 사람과의 차이(평균 0.290)도 1차(0.101)보다 크다. solar-decide v2는 0 판정 11개, 차이 0.284이고 047751-0017을 1.00으로 냈다
- 어느 판단 모델·지시 조합도 j_v8(사람과의 차이 0.078, 실패 클래스 5질의 중 047751-0017 0.00 포함)에 미치지 못했다. 실패 클래스 질의가 5개뿐이라 이 차이의 크기는 거칠게만 읽는다
- 2차 비용 약 $0.040(140요청). 1차와 합쳐 판단 모델 약 $0.056

## 결정

판단 모델은 평가 모듈에 넣지 않고 지금 judge(j_v8, DeepSeek V4.1 Flash)를 유지한다(사용자 결정 2026-10-26, D-48). 근거는 위 1·2차 관찰이다. `verdicts`에서 판단 모델은 지시를 바꿔도 사람이 낮게 준 질의를 j_v8만큼 가려내지 못했고, `item_handling`·`relevance`에서는 j_v8보다 나은 조합이 없었다. 다음 단계는 평가 모듈 동결이다.